# Project 4: Starter Notebook

This starter notebook contains some code that processes audio data and extracts features out of that data. It is not a necessary part for the rest of the project, but it provides good context of what each of the features represent and how audio is processed in the real world.

You do not have to submit this notebook.

**Setup:** You will need `librosa` to run this starter code. `librosa` is a Python library that contains several audio/signal processing functions that allows us to extract features from raw audio files. The `dsc80` environment does not have `librosa` installed. In your terminal, run the following before running anything in this notebook:
```sh
conda activate dsc80
pip install librosa
```

**File Setup:** Before running this notebook, make sure your project directory is structured as follows:

```
project4/
├── audio_exploration.ipynb   ← this notebook
├── your_notebook.ipynb       ← your main project notebook
└── Data/
    ├── music_tracks.csv
    ├── artists.csv
    └── audio_samples/
        ├── blues.wav
        ├── classical.wav
        ├── country.wav
        ├── disco.wav
        ├── hiphop.wav
        ├── jazz.wav
        ├── metal.wav
        ├── pop.wav
        ├── reggae.wav
        └── rock.wav
```

All paths in this notebook are relative to the project root. Make sure you open and run this notebook from that directory, not from inside `Data/`.

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

---
## Step 0: Audio Exploration

Before diving into the dataset, it helps to understand where features like `energy`, `tempo`, and `acousticness` actually come from. Music services like Spotify compute these automatically from the raw audio signal, but what does that mean in practice?

In this section we use a small subset of the Free Music Archive (FMA), a collection of Creative Commons-licensed tracks, to listen to real audio and extract a few features ourselves using `librosa`. The goal is to build intuition for the numbers you'll be analyzing for the rest of the project.


In [ ]:
import IPython.display as ipd
import os

# One 30-second sample per genre, from the GTZAN dataset (Creative Commons licensed)
audio_samples = {
    f.replace('.wav', ''): f'Data/audio_samples/{f}'
    for f in sorted(os.listdir('Data/audio_samples'))
    if f.endswith('.wav')
}

print(f'{len(audio_samples)} genres available:')
for genre, path in audio_samples.items():
    print(f'  {genre}: {path}')

In [ ]:
def extract_audio_features(url, duration=30):
    """Load up to `duration` seconds of audio from a URL and extract interpretable features."""
    y, sr = librosa.load(url, duration=duration, mono=True)

    # Higher BPM results in faster tempo, lower BPM result in lower tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    # A signal's root mean square (RMS) is directly associated with its energy or loudness
    loudness  = librosa.feature.rms(y=y).mean()

    # The spectral centroid is the weighted mean frequency of a signal
    # A higher centroid means there is more energy concentrated in higher frequencies, making the signal sound "brighter"
    brightness = librosa.feature.spectral_centroid(y=y, sr=sr).mean()

    # The zero crossing rate (ZCR) indicates how often an audio signal changes its sign
    # A higher ZCR represents harsher/more percusive sounds (e.g. drum hits, distorted guitars)
    # A lower ZCR indicates smoother sounds (e.g bass)
    roughness  = librosa.feature.zero_crossing_rate(y).mean()


    return {
        'tempo':      float(tempo),
        'loudness':   float(loudness),
        'brightness': float(brightness),
        'roughness':  float(roughness),
    }

In [ ]:
# Listen to any sample — change the genre to explore
EXAMPLE_GENRE = 'jazz'  # try any of the 10 genres available!

filepath = audio_samples[EXAMPLE_GENRE]
print(f'Playing: {filepath}')
ipd.display(ipd.Audio(filepath))

In [ ]:
# Extract features from the example track
features = extract_audio_features(filepath)
print(f'\nFeatures for {EXAMPLE_GENRE}:')
for k, v in features.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Compare features across all 10 genres
results = []
for genre, path in audio_samples.items():
    feats = extract_audio_features(path)
    feats['genre'] = genre
    results.append(feats)

audio_comparison = pd.DataFrame(results).set_index('genre')
audio_comparison.round(4)

**What do these features tell us?**

| Feature | What it measures | High value | Low value |
|---|---|---|---|
| `tempo` | Beats per minute | Fast (dance, metal) | Slow (ambient, classical) |
| `loudness` | RMS energy of the signal | Loud, intense tracks | Quiet, delicate tracks |
| `brightness` | Spectral centroid | Thin, bright, treble-heavy | Warm, bass-heavy, muffled |
| `roughness` | Zero-crossing rate | Noisy, harsh, percussive | Smooth, tonal, sustained |

These raw features are building blocks. Spotify's `energy`, `danceability`, and `acousticness` are higher-level interpretations computed from combinations of these and many more signal features using proprietary algorithms. The values you extracted above won't match the Spotify dataset exactly, but they encode similar information and point in the same direction.

---
## Extra: Visualizing the Audio Signal

You do not have to understand this section for the project. It shows what the audio signals actually look like and introduces the Mel spectrogram as a way to visualize frequency content over time.


#### Visualizing Audio with Mel Spectrograms

A **spectrogram** shows how the frequency content of a sound changes over time: the x-axis is time, the y-axis is frequency, and the color represents how loud each frequency is at each moment.

The **Mel** part refers to the frequency scale. Human hearing is not linear, we're much more sensitive to differences in low frequencies than high ones. The Mel scale compresses the high end and expands the low end to match how we actually perceive sound. This makes Mel spectrograms more useful for music analysis than raw spectrograms.

*Brighter colors = more energy at that frequency. Dark regions = silence or very low energy.*

Mel spectrograms are behind many real-world audio applications: Shazam, for instance, identifies songs by matching spectrogram "fingerprints" from a noisy recording against a database of clean references. They're also how platforms like YouTube and Spotify automatically detect whether a clip contains speech, music, or explicit content at scale.

In [ ]:
# Visualize the audio signal two ways:
# 1. Waveform: raw amplitude over time — shows loudness and rhythm structure
# 2. Mel spectrogram: frequency content over time — shows which pitches/frequencies are active
y, sr = librosa.load(filepath, duration=30, mono=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

librosa.display.waveshow(y, sr=sr, ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title(f"Waveform — {EXAMPLE_GENRE}")
axes[0].set_ylabel("Amplitude")

S_dB = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max)
img = librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=axes[1], cmap="magma")
axes[1].set_title(f"Mel Spectrogram — {EXAMPLE_GENRE}")
fig.colorbar(img, ax=axes[1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

In [ ]:
# Waveforms across all genres — notice how metal/rock are dense and chaotic,
# while classical and jazz have more structured, quieter passages
genres_list = list(audio_samples.items())
n = len(genres_list)

fig, axes = plt.subplots(n, 1, figsize=(12, 2 * n), sharex=True)

for ax, (genre, path) in zip(axes, genres_list):
    y, sr = librosa.load(path, duration=30, mono=True)
    librosa.display.waveshow(y, sr=sr, ax=ax, alpha=0.75)
    ax.set_ylabel(genre, fontsize=9, rotation=0, labelpad=55, va="center")
    ax.set_yticks([])

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Waveforms by Genre", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Mel spectrograms across all genres — each genre has a distinct frequency signature
cols = 2
rows = (n + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(14, 3 * rows))
axes = axes.flatten()

for ax, (genre, path) in zip(axes, genres_list):
    y, sr = librosa.load(path, duration=30, mono=True)
    S_dB = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=ax, cmap="magma")
    ax.set_title(genre)
    ax.set_xlabel("")
    ax.set_ylabel("")

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle("Mel Spectrograms by Genre", fontsize=13)
plt.tight_layout()
plt.show()